# Understanding Tools · MCP · A2A (Tools, MCP, A2A)
## LangChain Tools · MCP · Three A2A Patterns · Individual Tool Agent

> 📦 **Environment installation and run commands** are documented in [`env_guides/M02_3_mcp_a2a_eng.md`](env_guides/M02_3_mcp_a2a_eng.md).
> Repeated/common code has been factored out into the [`agentic_lib/`](agentic_lib) library (bootstrap · tools · mcp · a2a).

---

### Prerequisites
- Windows 11 + **CMD (`cmd.exe`)** + Python **3.11** (managed by `uv`), kernel = **`Agentic AI (uv)`**
- Verify the OpenAI-compatible LLM API connection first in ([`M02_1_local_llm_eng.ipynb`](M02_1_local_llm_eng.ipynb))
- Default LLM = **NVIDIA build + `deepseek-ai/deepseek-v4.1-flash`** (native and parallel tool calling); the web search tool requires network access (DuckDuckGo)

### Learning objectives
1. Define LangChain-based tools and connect them to an agent
2. Understand and implement the MCP (Model Context Protocol) architecture
3. Implement the three A2A (Agent-to-Agent) patterns (hierarchical / sequential / peer-to-peer)
4. Complete the final Individual Tool Agent

### Architecture overview
```
[User]
   ↓
[MCP Host: LangChain Agent]
   ├─[MCP Client] ──→ [MCP Server: search tool]
   ├─[MCP Client] ──→ [MCP Server: calculator tool]
   └─[LLM API: NVIDIA build / OpenRouter · OpenAI-compatible /v1]
```


In [1]:
# [setup] Self-contained setup — makes this notebook runnable on its own.
import sys, os
sys.path.insert(0, os.path.abspath(''))   # add notebooks/ to the import path

import utils
utils.reload_env()   # reload .env (refreshes LLM_PROVIDER, etc.) + print current provider status

from utils import get_llm, print_provider_status

# This part covers tools · MCP · A2A → uses mcp · a2a in addition to bootstrap · tools
from agentic_lib import bootstrap, tools, mcp, a2a
from agentic_lib.bootstrap import to_text   # provider-agnostic response normalization (including <think> removal)

# Packages needed for the tools exercise (including ddgs for web search). NVIDIA build and OpenRouter share LangChain's common interface
utils.uv_install(['langchain', 'langchain-openai', 'langchain-community',
                  'langchain-google-genai', 'langchain-anthropic',
                  'openai', 'requests', 'ddgs'])


llm = utils.get_llm()   # returns a LangChain BaseChatModel (default nvidia/deepseek-v4.1-flash)
print_provider_status()

LLM 공급자: nvidia
  NVIDIA build Key: 설정됨  /  Model: meta/llama-3.1-8b-instruct


[uv] 설치 완료: ['langchain', 'langchain-openai', 'langchain-community', 'langchain-google-genai', 'langchain-anthropic', 'openai', 'requests', 'ddgs']


LLM 공급자: nvidia
  NVIDIA build Key: 설정됨  /  Model: meta/llama-3.1-8b-instruct


---
## 4. Defining LangChain Tools

Define the tools the agent can use.

In [2]:
# §4 Tools — reuse the common tools (agentic_lib.tools) + real web search (DuckDuckGo)
from langchain_community.tools import DuckDuckGoSearchRun

# Real web search tool (network required). If offline, you can substitute tools.search_web (simulation).
search = DuckDuckGoSearchRun()

# Use the common @tool functions from agentic_lib.tools as is (calculator / get_current_time / get_weather)
agent_tools = [tools.calculator, tools.get_current_time, tools.get_weather, search]

print("Registered tools:")
for t in agent_tools:
    print(f"  - {t.name}: {t.description[:60]}...")

등록된 도구:
  - calculator: 수학 수식을 계산합니다. 예: '2 ** 10', 'sqrt(144)', '(3+4)*5'.

    Pyt...
  - get_current_time: 현재 날짜와 시간을 'YYYY-MM-DD HH:MM:SS' 형식으로 반환합니다.

    인자가 없는 도구를...
  - get_weather: 도시의 현재 날씨를 조회합니다(무료 Open-Meteo, API 키 불필요). 예: '서울', '부산'.

...
  - duckduckgo_search: A wrapper around DuckDuckGo Search. Useful for when you need...


C:\Users\stshin\AppData\Local\Temp\ipykernel_40712\4088401716.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


---
## 5. MCP (Model Context Protocol) Architecture

### Core MCP components
- **MCP Host:** the AI application (LangChain Agent)
- **MCP Client:** the component that maintains the connection to a server
- **MCP Server:** a program that provides context (tools/data)

```
[MCP Host]
  └─[MCP Client 1] ↔ [MCP Server: file system]
  └─[MCP Client 2] ↔ [MCP Server: web search]
  └─[MCP Client 3] ↔ [MCP Server: database]
```

In [3]:
# The MCP infrastructure (MCPServer/Client/Host, Calculator/FileSystem servers) has been moved to agentic_lib.mcp.
from agentic_lib.mcp import MCPHost, CalculatorMCPServer, FileSystemMCPServer

print("=== MCP system configuration ===")
host = MCPHost()
host.connect(CalculatorMCPServer())
host.connect(FileSystemMCPServer())

print("\n=== Available tools ===")
for server, tools_list in host.list_all_tools().items():
    print(f"\n[{server}]")
    for t in tools_list:
        print(f"  - {t['name']}: {t['description']}")

print("\n=== MCP tool call test ===")
print("  Calculation result:", host.call_tool("calculator-server", "calculate", expression="2 ** 10"))
print("  File write:", host.call_tool("filesystem-server", "write_file", path="memo.txt", content="MCP test file"))
print("  File read:", host.call_tool("filesystem-server", "read_file", path="memo.txt"))

=== MCP 시스템 구성 ===
[MCP Client] 'calculator-server' 서버에 연결됨
[MCP Client] 'filesystem-server' 서버에 연결됨

=== 사용 가능한 도구 목록 ===

[calculator-server]
  - calculate: 수학 계산 수행
  - factorial: 팩토리얼 계산

[filesystem-server]
  - read_file: 파일 읽기
  - write_file: 파일 쓰기
  - list_files: 파일 목록 조회

=== MCP 도구 호출 테스트 ===
  [MCP] calculator-server.calculate({'expression': '2 ** 10'})
  계산 결과: {'result': 1024, 'expression': '2 ** 10'}
  [MCP] filesystem-server.write_file({'path': 'memo.txt', 'content': 'MCP 테스트 파일'})
  파일 쓰기: {'success': True, 'path': 'memo.txt'}
  [MCP] filesystem-server.read_file({'path': 'memo.txt'})
  파일 읽기: MCP 테스트 파일


---
## 6. Three A2A (Agent-to-Agent) Patterns

### Pattern 1: Hierarchical
A lead agent distributes work to sub-agents

In [4]:
# The A2A infrastructure (BaseAgent/CoordinatorAgent/PipelineAgent/SharedCanvas/PeerAgent) has been moved to agentic_lib.a2a.
from agentic_lib.a2a import BaseAgent, CoordinatorAgent

# Specialist sub-agents
search_agent = BaseAgent("SearchAgent", "search specialist")
calc_agent = BaseAgent("CalculatorAgent", "math specialist")
file_agent = BaseAgent("FileAgent", "file management specialist")

coordinator = CoordinatorAgent("Coordinator", [search_agent, calc_agent, file_agent])

print("=== Hierarchical A2A pattern ===")
print("The lead agent distributes tasks to sub-agents\n")
task = "Search for AI trends, calculate the budget 1000+500, and save the results to a file"
print(f"Input task: {task}\n")
results = coordinator.delegate(task)
print(f"\nFinal result: {results}")

=== 계층형(Hierarchical) A2A 패턴 ===
대장 에이전트가 작업을 하위 에이전트에 분배

입력 태스크: AI 트렌드를 검색하고, 예산 1000+500을 계산하고, 결과를 파일에 저장해줘

  [Coordinator] → [SearchAgent]: 검색 태스크: AI 트렌드를 검색하고, 예산 1000+500을 계산하고, 결과를 파일에 저장해줘
  [Coordinator] → [CalculatorAgent]: 계산 태스크: AI 트렌드를 검색하고, 예산 1000+500을 계산하고, 결과를 파일에 저장해줘
  [Coordinator] → [FileAgent]: 파일 태스크: AI 트렌드를 검색하고, 예산 1000+500을 계산하고, 결과를 파일에 저장해줘

최종 결과: {'search': "[SearchAgent] '검색 태스크: AI 트렌드를 검색하고, 예산 1000+500을 계산하고, 결과를 파일에 저' 처리 완료", 'calc': "[CalculatorAgent] '계산 태스크: AI 트렌드를 검색하고, 예산 1000+500을 계산하고, 결과를 파일에 저' 처리 완료", 'file': "[FileAgent] '파일 태스크: AI 트렌드를 검색하고, 예산 1000+500을 계산하고, 결과를 파일에 저' 처리 완료"}


In [5]:
# Pattern 2: Sequential (Pipeline) — PipelineAgent lives in agentic_lib.a2a
from agentic_lib.a2a import PipelineAgent

# Processing function for each pipeline stage (for demo purposes — in practice, LLM/tool calls)
def collect_data(topic):
    """Data collection stage."""
    return {"topic": topic,
            "raw_data": f"Raw data about {topic}: [article1, article2, article3]",
            "source_count": 3}

def analyze_data(data):
    """Data analysis stage."""
    data["analysis"] = f"Extracted key trends from {data['source_count']} sources"
    data["keywords"] = ["AI", "Agent", "LLM"]
    return data

def generate_report(data):
    """Report generation stage."""
    return (f"=== {data['topic']} Analysis Report ===\n"
            f"Analysis: {data['analysis']}\n"
            f"Keywords: {', '.join(data['keywords'])}")

def review_report(report):
    """Report review stage."""
    return report + "\n[Review complete] Passed quality check"

# Build and connect the pipeline (set_next returns the next agent → chaining)
collector = PipelineAgent("Collector", "Data collection", collect_data)
analyzer = PipelineAgent("Analyzer", "Data analysis", analyze_data)
generator = PipelineAgent("Generator", "Report generation", generate_report)
reviewer = PipelineAgent("Reviewer", "Report review", review_report)
collector.set_next(analyzer).set_next(generator).set_next(reviewer)

print("=== Sequential A2A pattern ===")
print("Processed in order, like a conveyor belt\n")
final_result = collector.execute("Agentic AI trends")
print(f"\nFinal report:\n{final_result}")

=== 순차형(Sequential) A2A 패턴 ===
컨베이어 벨트처럼 순서대로 처리

  [Collector(데이터 수집)] 처리 중: Agentic AI 트렌드
    → 출력: {'topic': 'Agentic AI 트렌드', 'raw_data': 'Agentic AI 트렌드에 관한 원시 데이터: [기사1, 기사2, 기
  [Analyzer(데이터 분석)] 처리 중: {'topic': 'Agentic AI 트렌드', 'raw_data': 'Agentic AI 트렌드에 관한 
    → 출력: {'topic': 'Agentic AI 트렌드', 'raw_data': 'Agentic AI 트렌드에 관한 원시 데이터: [기사1, 기사2, 기
  [Generator(보고서 생성)] 처리 중: {'topic': 'Agentic AI 트렌드', 'raw_data': 'Agentic AI 트렌드에 관한 
    → 출력: === Agentic AI 트렌드 분석 보고서 ===
분석: 3개 소스에서 주요 트렌드 추출 완료
키워드: AI, 에이전트, LLM
  [Reviewer(보고서 검토)] 처리 중: === Agentic AI 트렌드 분석 보고서 ===
분석: 3개 소스에서 주요 트렌드 추출 완료
키워드: 
    → 출력: === Agentic AI 트렌드 분석 보고서 ===
분석: 3개 소스에서 주요 트렌드 추출 완료
키워드: AI, 에이전트, LLM
[검토 완료

최종 보고서:
=== Agentic AI 트렌드 분석 보고서 ===
분석: 3개 소스에서 주요 트렌드 추출 완료
키워드: AI, 에이전트, LLM
[검토 완료] 품질 검증 통과


In [6]:
# Pattern 3: Peer-to-Peer (Shared Canvas) — SharedCanvas/PeerAgent live in agentic_lib.a2a
from agentic_lib.a2a import SharedCanvas, PeerAgent

# Generate opinions per area of expertise (demo rules — in practice, inject a closure that calls an LLM). (expertise, topic) -> str
def opinion(expertise, topic):
    """Return an opinion string for the given area of expertise (simulation)."""
    book = {
        "Technology": f"For {topic}, the optimal tech stack is LangGraph + an OpenAI-compatible LLM API.",
        "Business": f"The business value of {topic} is a 30% annual cost reduction through work automation.",
        "Security": f"When implementing {topic}, input/output filtering with NeMo Guardrails is essential.",
        "UX": f"The user experience of {topic} should target a response time of < 2 seconds.",
    }
    return book.get(expertise, f"{expertise} perspective on {topic}")

# Shared canvas + peer agents (inject the opinion generator via opinion_fn)
canvas = SharedCanvas()
peers = [
    PeerAgent("Tech Expert", "Technology", canvas, opinion_fn=opinion),
    PeerAgent("Biz Expert", "Business", canvas, opinion_fn=opinion),
    PeerAgent("Security Expert", "Security", canvas, opinion_fn=opinion),
    PeerAgent("UX Expert", "UX", canvas, opinion_fn=opinion),
]

print("=== Peer-to-Peer A2A pattern ===")
print("Equal agents collaborate on a shared canvas\n")
topic = "Plan for adopting AI agents within the company"
print(f"Discussion topic: {topic}\n")
for agent in peers:
    agent.contribute(topic)
print(f"\n{canvas.get_summary()}")

=== 수평형(Peer-to-Peer) A2A 패턴 ===
동등한 에이전트들이 공유 캔버스에서 협업

토론 주제: 회사 내부 AI 에이전트 도입 방안

  [Tech Expert] 캔버스에 '기술' 의견 추가
  [Biz Expert] 캔버스에 '비즈니스' 의견 추가
  [Security Expert] 캔버스에 '보안' 의견 추가
  [UX Expert] 캔버스에 'UX' 의견 추가

[공유 캔버스 현황]
  [Tech Expert] 기술_의견: 회사 내부 AI 에이전트 도입 방안의 기술 스택은 LangGraph + 로컬 LLM(OpenAI 호환)이 최
  [Biz Expert] 비즈니스_의견: 회사 내부 AI 에이전트 도입 방안의 비즈니스 가치는 업무 자동화로 연간 30% 비용 절감입니다.
  [Security Expert] 보안_의견: 회사 내부 AI 에이전트 도입 방안 구현 시 NeMo Guardrails 로 입출력 필터링이 필수입니다.
  [UX Expert] UX_의견: 회사 내부 AI 에이전트 도입 방안의 사용자 경험은 응답 속도 < 2초를 목표로 해야 합니다.


---
## 7. Implementing a Real LangChain-Based Agent

Connect the tools defined above to a LangChain Agent.

> **Tool calling provider** — The default, **NVIDIA build + `deepseek-ai/deepseek-v4.1-flash`**, reliably produces native `tool_calls` and also supports parallel calls, so the cell below uses the default provider returned by `utils.get_llm()` as is for the agent. If you change `.env` to `LLM_PROVIDER=openrouter`, the same code also works with OpenRouter (`openrouter/free`).
>
> 📓 **Compare it yourself with repeated experiments**: the function calling reliability and parallel calls of NVIDIA `deepseek-v4.1-flash` vs OpenRouter small (`lfm-2.5-2.6b`), large (`nemotron-3-super`), and router (`openrouter/free`) are compared over N repeated runs → supplementary notebook **`M02_2_function_calling_eng.ipynb`**. **Key point**: tool-calling quality is not determined by model size alone; it also depends on model characteristics and provider availability, so you should measure the model you plan to use yourself.

In [7]:
from langchain.agents import create_agent

system_prompt = """You are a capable AI agent that can use a variety of tools.
Analyze the user's request, choose the appropriate tools, and provide an accurate answer.

Available tools:
- calculator: math calculations
- get_current_time: current date/time
- get_weather: weather lookup
- duckduckgo_search: web search

Always answer in English."""

try:
    if llm is None:
        print("LLM connection required: check the provider/key in .env.")
    else:
        # The default provider (nvidia/deepseek-v4.1-flash) has stable native tool calling → use get_llm as is
        agent_llm = utils.get_llm()
        agent = create_agent(agent_llm, agent_tools, system_prompt=system_prompt)

        print("=== Running the LangChain agent ===")
        result = agent.invoke({
            "messages": [("human", "Tell me today's date, and check the weather in Seoul. Also, calculate the square root of 100.")]
        })
        final = to_text(result["messages"][-1].content)
        print(f"\nFinal answer: {final}")

except Exception as e:
    print(f"Agent execution error: {type(e).__name__}: {e}")
    print("Check the LLM backend/tool calling settings.")

=== LangChain 에이전트 실행 ===


에이전트 실행 오류: Exception: [500] {'message': 'Failed to generate completions: Failed to apply prompt template: invalid operation: This model only supports single tool-calls at once! (in tool_use:95)', 'type': 'Internal Server Error', 'code': 500}
{'error': {'message': 'Failed to generate completions: Failed to apply prompt template: invalid operation: This model only supports single tool-calls at once! (in tool_use:95)', 'type': 'Internal Server Error', 'code': 500}, 'status': 500}
LLM 백엔드/도구 호출 설정을 확인하세요.


---
## 8. Final Exercise: Implementing an Individual Tool Agent

### Assignment: build your own specialist agent
Use the template below to implement an agent specialized for a particular domain.

In [8]:
from langchain.tools import tool
from langchain.agents import create_agent
import math
from datetime import datetime

# ============================================================
# TODO: Add your own tools here!
# ============================================================

@tool
def my_custom_tool(input: str) -> str:
    """Write the tool description here.
    Clearly explain what input it takes and what output it returns.
    """
    return f"My tool result: {input}"


@tool
def get_stock_price(symbol: str) -> str:
    """Look up the current price of a stock ticker.
    e.g., 'AAPL', 'GOOGL', 'NVDA'
    """
    mock_prices = {
        "AAPL": 182.5, "GOOGL": 175.3, "NVDA": 875.2,
        "MSFT": 415.7, "AMZN": 188.4,
    }
    price = mock_prices.get(symbol.upper(), 100.0)
    return f"{symbol.upper()} current price: ${price:.2f} USD (simulated)"


@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert currencies. Supported currencies: USD, KRW, EUR, JPY"""
    rates = {
        "USD_KRW": 1340.0, "USD_EUR": 0.92, "USD_JPY": 149.5,
        "KRW_USD": 1/1340.0, "EUR_USD": 1/0.92,
    }
    key = f"{from_currency.upper()}_{to_currency.upper()}"
    rate = rates.get(key, 1.0)
    result = amount * rate
    return f"{amount} {from_currency} = {result:.2f} {to_currency} (exchange rate: {rate})"


@tool
def calculator_fin(expression: str) -> str:
    """Perform math/financial calculations. Enter a Python expression."""
    try:
        result = eval(expression, {"__builtins__": {}}, {"math": math})
        return f"{expression} = {result}"
    except Exception as e:
        return f"Error: {e}"


finance_tools = [get_stock_price, convert_currency, calculator_fin]

finance_system_prompt = """You are an AI agent specialized in finance.
You help with stock price lookups, currency conversion, return calculations, and more.
Always answer in English with accurate numbers."""

try:
    if llm is None:
        print("LLM connection required: set the API key in .env.")
    else:
        # The default provider (nvidia/deepseek-v4.1-flash) has stable native tool calling → use get_llm as is
        agent_llm = utils.get_llm()
        finance_agent = create_agent(agent_llm, finance_tools, system_prompt=finance_system_prompt)

        print("=== Running the finance specialist agent ===")
        result = finance_agent.invoke({
            "messages": [("human", "Tell me the NVIDIA stock price, and convert 100 dollars to Korean won. What is the total value of 10 NVIDIA shares?")]
        })
        final = to_text(result["messages"][-1].content)
        print(f"\nFinal answer:\n{final}")

except Exception as e:
    print(f"Agent error: {type(e).__name__}: {e}")

=== 금융 전문 에이전트 실행 ===


에이전트 오류: Exception: [500] {'message': 'Failed to generate completions: Failed to apply prompt template: invalid operation: This model only supports single tool-calls at once! (in tool_use:95)', 'type': 'Internal Server Error', 'code': 500}
{'error': {'message': 'Failed to generate completions: Failed to apply prompt template: invalid operation: This model only supports single tool-calls at once! (in tool_use:95)', 'type': 'Internal Server Error', 'code': 500}, 'status': 500}


---
## 9. Summary and Preview of the Next Module

### Key takeaways
1. **LLM API:** NVIDIA build (default) and OpenRouter both provide an **OpenAI-compatible `/v1`** → used with the same code, switched with a single `LLM_PROVIDER` line
2. **MCP:** connects agents to external tools/data in a standard way via the Host-Client-Server structure
3. **A2A patterns:**
   - **Hierarchical:** a lead agent distributes work → centralized
   - **Sequential:** step-by-step processing in a pipeline → suited to data transformation
   - **Peer-to-peer:** collaboration on a shared canvas → suited to creative work
4. **LangChain Agent:** a practical agent combining tools + LLM + prompt (deepseek-v4.1-flash native and parallel tool calling)

---
### References
- Environment guide: [`env_guides/M02_3_mcp_a2a_eng.md`](env_guides/M02_3_mcp_a2a_eng.md)
- Common library: `agentic_lib` (bootstrap · tools · mcp · a2a)
- Official MCP documentation: https://modelcontextprotocol.io
- LangChain agents guide: https://python.langchain.com/docs/how_to/#agents